In [ ]:
import torch
import torch.nn as nn

from torchvision import models, transforms, datasets
from PIL import Image

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
dataset_path = "../data/raw"

dataset = datasets.ImageFolder(
    root=dataset_path
)

class_names = dataset.classes

print("Classes:", len(class_names))
print(class_names)

Classes: 15
['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [ ]:
model = models.efficientnet_b0(
    weights=None
)

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(1280, 15)
)

model = model.to(device)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [ ]:
model.load_state_dict(
    torch.load(
        "../models/efficientnet_b0_99_39.pth",
        map_location=device
    )
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    image_tensor = test_transform(image)

    image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(device)

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        top_probs, top_indices = torch.topk(
            probabilities,
            k=3
        )

    print("Top 3 Predictions:\n")

    for i in range(3):

        disease = class_names[
            top_indices[0][i].item()
        ]

        confidence = (
            top_probs[0][i].item() * 100
        )

        print(
            f"{i+1}. {disease} : {confidence:.2f}%"
        )

In [ ]:
import os

folder = "../data/raw"

for root, dirs, files in os.walk(folder):
    for file in files[:3]:
        print(os.path.join(root, file))
        break

../data/raw\Pepper__bell___Bacterial_spot\0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Spot 8964.JPG
../data/raw\Pepper__bell___healthy\00100ffa-095e-4881-aebf-61fe5af7226e___JR_HL 7886.JPG
../data/raw\Potato___Early_blight\001187a0-57ab-4329-baff-e7246a9edeb0___RS_Early.B 8178.JPG
../data/raw\Potato___healthy\00fc2ee5-729f-4757-8aeb-65c3355874f2___RS_HL 1864.JPG
../data/raw\Potato___Late_blight\0051e5e8-d1c4-4a84-bf3a-a426cdad6285___RS_LB 4640.JPG
../data/raw\Tomato_Bacterial_spot\00416648-be6e-4bd4-bc8d-82f43f8a7240___GCREC_Bact.Sp 3110.JPG
../data/raw\Tomato_Early_blight\0012b9d2-2130-4a06-a834-b1f3af34f57e___RS_Erly.B 8389.JPG
../data/raw\Tomato_healthy\000146ff-92a4-4db6-90ad-8fce2ae4fddd___GH_HL Leaf 259.1.JPG
../data/raw\Tomato_Late_blight\0003faa8-4b27-4c65-bf42-6d9e352ca1a5___RS_Late.B 4946.JPG
../data/raw\Tomato_Leaf_Mold\00694db7-3327-45e0-b4da-a8bb7ab6a4b7___Crnl_L.Mold 6923.JPG
../data/raw\Tomato_Septoria_leaf_spot\002533c1-722b-44e5-9d2e-91f7747b2543___Keller.St_CG 1831.JP

In [ ]:
import matplotlib.pyplot as plt

def show_image(image_path):

    image = Image.open(image_path)

    plt.imshow(image)

    plt.axis("off")

    plt.show()

In [ ]:
predict_image(
    r"../data/raw\Tomato_healthy\000146ff-92a4-4db6-90ad-8fce2ae4fddd___GH_HL Leaf 259.1.JPG"
)

Top 3 Predictions:

1. Tomato_healthy : 99.99%
2. Tomato_Late_blight : 0.01%
3. Tomato_Bacterial_spot : 0.00%
